# pathlib module
- The Python `pathlib` module provides an object-oriented approach to handling file system paths. It offers classes to handle different types of paths, such as POSIX and Windows, making path manipulations more intuitive and less error-prone.
- `Path` = a concrete path you can actually use on disk (create, read, delete...). **This is the one you use 99% of the time.**
- `PurePath` = only string-like path manipulation, never touches the disk.
- Modern Python code prefers `pathlib` over `os.path`.

In [ ]:
from pathlib import Path

### 1. Creating Path objects
- `Path("some/path")`, `Path.cwd()`, `Path.home()`
- Join paths with the **`/` operator** ⭐ (instead of `os.path.join`).

In [ ]:
print(Path.cwd())
print(Path.home())

p = Path("playground") / "data" / "raw" / "train.csv"
print(p, type(p))

### 2. Path parts (no disk access)
| Attribute | Result for `playground/data/raw/train.csv` |
|---|---|
| `.name` | `'train.csv'` |
| `.stem` | `'train'` |
| `.suffix` | `'.csv'` |
| `.parent` | `playground/data/raw` |
| `.parents` | all ancestors |
| `.parts` | `('playground', 'data', 'raw', 'train.csv')` |
| `.with_suffix(".json")` | `playground/data/raw/train.json` |
| `.with_name("test.csv")` | `playground/data/raw/test.csv` |
| `.resolve()` | absolute path |

In [ ]:
print(p.name)
print(p.stem)
print(p.suffix)
print(p.parent)
print(list(p.parents))
print(p.parts)
print(p.with_suffix(".json"))
print(p.with_name("test.csv"))
print(p.resolve())

### 3. Creating directories & files
- `path.mkdir(parents=True, exist_ok=True)` ⭐ — like `os.makedirs`.
- `path.touch()` — create an empty file.
- Note: these are **methods on a Path object**, so call `Path("test").mkdir()`, not `pathlib.Path.mkdir("test")`.

In [ ]:
p.parent.mkdir(parents=True, exist_ok=True)   # creates playground/data/raw
Path("playground/notes.txt").touch()
print(list(Path("playground").iterdir()))

### 4. Reading & writing files (no `open()` needed)
- `write_text()` / `read_text()` for text, `write_bytes()` / `read_bytes()` for binary.
- `path.open()` also works for big files or line-by-line reading.

In [ ]:
p.write_text("x,y\n1,2\n3,4\n")
print(p.read_text())

with p.open() as f:
    for line in f:
        print(line.strip())

### 5. Checking paths
`exists()`, `is_file()`, `is_dir()`, `stat().st_size`

In [ ]:
print(p.exists(), p.is_file(), p.is_dir())
print(p.parent.is_dir())
print(p.stat().st_size, "bytes")

### 6. Listing & searching ⭐
| Method | What it does |
|---|---|
| `path.iterdir()` | direct children (like `os.listdir`) |
| `path.glob("*.csv")` | match pattern in this folder |
| `path.glob("**/*.csv")` / `path.rglob("*.csv")` | match pattern **recursively** (like `os.walk` + filter) |

`glob` is called on a Path object: `Path("/usr").glob("*")`, not `pathlib.Path.glob("/usr/")`. It returns a generator → wrap in `list()` or loop.

In [ ]:
for name in ["a.txt", "b.txt", "data/raw/test.csv"]:
    (Path("playground") / name).write_text("hello")

for child in Path("playground").iterdir():
    print(child, "dir" if child.is_dir() else "file")

In [ ]:
print(list(Path("playground").glob("*.txt")))    # only top level
print(list(Path("playground").rglob("*.csv")))   # recursive

In [ ]:
# common ML pattern: collect all files of a dataset
files = sorted(Path("playground").rglob("*.csv"))
for f in files:
    print(f.name, f.stat().st_size)

### 7. Rename / delete
| Method | What it does |
|---|---|
| `path.rename(target)` | rename / move, returns the new Path |
| `path.unlink()` | delete a file (`missing_ok=True` to avoid error) |
| `path.rmdir()` | delete an **empty** directory |
| `shutil.rmtree(path)` | delete a non-empty directory |

In [ ]:
new = Path("playground/a.txt").rename("playground/a_renamed.txt")
print(new)

Path("playground/b.txt").unlink()
Path("playground/does_not_exist.txt").unlink(missing_ok=True)  # no error

Path("playground/empty").mkdir()
Path("playground/empty").rmdir()

print(sorted(Path("playground").iterdir()))

In [ ]:
# cleanup
import shutil
shutil.rmtree("playground")
print(Path("playground").exists())

### PurePath
Only manipulates paths as text — never touches disk. Useful to handle paths of another OS (e.g. Windows paths on Linux).

In [ ]:
from pathlib import PurePath, PureWindowsPath

print(PurePath("a/b/c.txt").suffix)
w = PureWindowsPath(r"C:\Users\me\data.csv")
print(w.name, w.drive, w.parent)

## os vs pathlib cheat sheet
| Task | `os` / `os.path` | `pathlib` |
|---|---|---|
| current dir | `os.getcwd()` | `Path.cwd()` |
| home dir | `os.path.expanduser("~")` | `Path.home()` |
| join | `os.path.join(a, b)` | `Path(a) / b` |
| exists | `os.path.exists(p)` | `p.exists()` |
| is file / dir | `os.path.isfile(p)` / `os.path.isdir(p)` | `p.is_file()` / `p.is_dir()` |
| file name | `os.path.basename(p)` | `p.name` |
| parent dir | `os.path.dirname(p)` | `p.parent` |
| extension | `os.path.splitext(p)[1]` | `p.suffix` |
| absolute | `os.path.abspath(p)` | `p.resolve()` |
| make dirs | `os.makedirs(p, exist_ok=True)` | `p.mkdir(parents=True, exist_ok=True)` |
| list dir | `os.listdir(p)` | `p.iterdir()` |
| find recursively | `os.walk(p)` + filter | `p.rglob("*.txt")` |
| rename | `os.rename(a, b)` | `p.rename(b)` |
| delete file | `os.remove(p)` | `p.unlink()` |
| delete empty dir | `os.rmdir(p)` | `p.rmdir()` |
| read file | `open(p).read()` | `p.read_text()` |
| env variable | `os.getenv("KEY")` | — (use `os`) |

**Rule of thumb:** use `pathlib` for paths & files, `os` for environment variables and system stuff.

## 🏋️ Exercises
- Try each exercise yourself in the empty cell first, then open **💡 Show solution** to compare.
- Run the **setup** cell first — it creates a practice folder with sample files.
- If something breaks, just re-run the setup cell to reset everything.

In [ ]:
# SETUP: creates a practice folder (re-run anytime to reset)
from pathlib import Path
import shutil

base = Path("ex_pathlib")
shutil.rmtree(base, ignore_errors=True)
sample = {
    "readme.txt": "practice folder",
    "downloads/report.pdf": "pdf" * 100,
    "downloads/photo1.jpg": "img" * 30,
    "downloads/photo2.png": "img" * 10,
    "downloads/sales.csv": "a,b\n1,2\n",
    "downloads/notes.txt": "some notes",
    "datasets/2024/jan.csv": "a,b\n" + "1,2\n" * 50,
    "datasets/2024/feb.csv": "a,b\n" + "1,2\n" * 20,
    "datasets/2025/mar.csv": "a,b\n1,2\n",
}
for rel, content in sample.items():
    p = base / rel
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(content)
print("setup done:", sorted(p.name for p in base.iterdir()))

### Exercise 1
**Path parts** — given `p = Path("datasets/images/cat_001.jpg")` print:
its name, stem, extension, parent folder, all parts, and a new path with `.png` instead of `.jpg`.

In [ ]:
# your code here

<details>
<summary>💡 Show solution</summary>

```python
p = Path("datasets/images/cat_001.jpg")
print(p.name)
print(p.stem)
print(p.suffix)
print(p.parent)
print(p.parts)
print(p.with_suffix(".png"))
```
</details>

### Exercise 2
**Build paths with `/`** — using `Path.home()` and the `/` operator, build the path `~/projects/ml/data/train.csv` and print it.
Then print only the folder part of it (no file name).

In [ ]:
# your code here

<details>
<summary>💡 Show solution</summary>

```python
p = Path.home() / "projects" / "ml" / "data" / "train.csv"
print(p)
print(p.parent)
```
</details>

### Exercise 3
**Write & read** — create `ex_pathlib/config.txt` containing these 3 lines:
```
lr=0.001
epochs=10
batch_size=32
```
Then read it back and turn it into a dict: `{'lr': '0.001', 'epochs': '10', 'batch_size': '32'}`.

In [ ]:
# your code here

<details>
<summary>💡 Show solution</summary>

```python
cfg = base / "config.txt"
cfg.write_text("lr=0.001\nepochs=10\nbatch_size=32\n")

config = {}
for line in cfg.read_text().splitlines():
    key, value = line.split("=")
    config[key] = value
print(config)
```
</details>

### Exercise 4
**Find all CSV files** — find every `.csv` file inside `ex_pathlib/` (any depth), sorted, and print each file's name with its size in bytes.
Then print the total size of all of them.

In [ ]:
# your code here

<details>
<summary>💡 Show solution</summary>

```python
csv_files = sorted(base.rglob("*.csv"))
total = 0
for f in csv_files:
    size = f.stat().st_size
    total += size
    print(f.name, size)
print("total:", total)
```
</details>

### Exercise 5
**Group by year** — the CSVs in `ex_pathlib/datasets/` are inside year folders. Build a dict `{year: [file names]}`, e.g. `{'2024': ['feb.csv', 'jan.csv'], '2025': ['mar.csv']}`.
Hint: `f.parent.name`

In [ ]:
# your code here

<details>
<summary>💡 Show solution</summary>

```python
by_year = {}
for f in sorted((base / "datasets").rglob("*.csv")):
    by_year.setdefault(f.parent.name, []).append(f.name)
print(by_year)
```
</details>

### Exercise 6
**Organize downloads** — move every file in `ex_pathlib/downloads/` into a sub-folder named after its extension:
```
downloads/
├── csv/sales.csv
├── jpg/photo1.jpg
├── pdf/report.pdf
├── png/photo2.png
└── txt/notes.txt
```
Hint: `suffix`, `mkdir(exist_ok=True)`, `rename`. Careful: only move **files**, not the new folders!

In [ ]:
# your code here

<details>
<summary>💡 Show solution</summary>

```python
downloads = base / "downloads"
for f in list(downloads.iterdir()):
    if f.is_file():
        folder = downloads / f.suffix.lstrip(".")
        folder.mkdir(exist_ok=True)
        f.rename(folder / f.name)

for f in sorted(downloads.rglob("*")):
    print(f.relative_to(downloads))
```
</details>

### Exercise 7
**os → pathlib** — rewrite this code using only `pathlib` (it should do exactly the same):
```python
import os
folder = os.path.join("ex_pathlib", "backup")
if not os.path.exists(folder):
    os.makedirs(folder)
path = os.path.join(folder, "log.txt")
with open(path, "w") as f:
    f.write("backup done")
print(os.path.basename(path), os.path.getsize(path))
```

In [ ]:
# your code here

<details>
<summary>💡 Show solution</summary>

```python
folder = Path("ex_pathlib") / "backup"
folder.mkdir(parents=True, exist_ok=True)
path = folder / "log.txt"
path.write_text("backup done")
print(path.name, path.stat().st_size)
```
</details>

### Cleanup
Run this when you're done to delete the practice folder.

In [ ]:
shutil.rmtree("ex_pathlib", ignore_errors=True)
print(Path("ex_pathlib").exists())